# Address AI: Full-Data GPU Training (Colab)
Use this notebook to train [2_finetune_t5.py](2_finetune_t5.py) on full data with CUDA, using [data/address_training_kier_v1_strict.jsonl](data/address_training_kier_v1_strict.jsonl).
It includes an automatic batch-size fallback to handle GPU out-of-memory issues.

## 1. Runtime + Dependencies
In Colab, select: Runtime -> Change runtime type -> Hardware accelerator -> GPU.
Then run the next cell to verify GPU and install packages.

In [ ]:
!nvidia-smi
!python --version
!pip install -U pip
!pip install -r requirements.txt

## 2. Mount Drive + Set Project Path
Place your project folder in Google Drive (example: MyDrive/address_ai1).
The next cell mounts Drive, changes directory, and checks required files.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/address_ai1'  # change if needed
os.chdir(PROJECT_DIR)

print('Working dir:', os.getcwd())
print('Has training script:', os.path.exists('2_finetune_t5.py'))
print('Has strict dataset:', os.path.exists('data/address_training_kier_v1_strict.jsonl'))

## 3. Train on Full Data (CUDA)
This cell runs full-data training and auto-retries with lower batch size if memory is insufficient.

In [ ]:
import subprocess
import pathlib
import re

src = pathlib.Path('2_finetune_t5.py')
assert src.exists(), '2_finetune_t5.py not found'
assert pathlib.Path('data/address_training_kier_v1_strict.jsonl').exists(), 'strict dataset not found'

def make_temp_script(batch_size: int) -> pathlib.Path:
    txt = src.read_text(encoding='utf-8')
    txt = re.sub(r'^BATCH_SIZE\s*=\s*\d+', f'BATCH_SIZE   = {batch_size}', txt, flags=re.M)
    tmp = pathlib.Path(f'2_finetune_t5_bs{batch_size}.py')
    tmp.write_text(txt, encoding='utf-8')
    return tmp

batch_candidates = [64, 48, 32, 24, 16, 8]
success = False

for bs in batch_candidates:
    print(f'\nTrying batch size: {bs}')
    script_path = make_temp_script(bs)
    cmd = [
        'python', str(script_path),
        '--device', 'cuda',
        '--full-data',
        '--train-jsonl', 'data/address_training_kier_v1_strict.jsonl'
    ]
    proc = subprocess.run(cmd)
    if proc.returncode == 0:
        print(f'\nTraining completed successfully with batch size {bs}')
        success = True
        break
    print(f'Run failed with batch size {bs}, trying lower batch...')

if not success:
    raise RuntimeError('All batch sizes failed. Check logs above.')

## 4. Save Trained Model to Drive
This cell copies [models/t5_address](models/t5_address) to Google Drive with a timestamp.

In [ ]:
import os
import shutil
import time
from datetime import datetime

model_dir = 'models/t5_address'
backup_root = '/content/drive/MyDrive'
timeout_seconds = 600
poll_seconds = 15

start = time.time()
while not os.path.isdir(model_dir) and (time.time() - start) < timeout_seconds:
    elapsed = int(time.time() - start)
    print(f'Waiting for training output... {elapsed}s elapsed')
    time.sleep(poll_seconds)

if not os.path.isdir(model_dir):
    print(f'{model_dir} not found after waiting {timeout_seconds}s.')
    print('Run Cell 3 again and wait until it prints "Training complete" and "Saved best model".')
else:
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    backup_dir = os.path.join(backup_root, f'address_ai1_trained_t5_address_{stamp}')
    shutil.copytree(model_dir, backup_dir)
    print('Saved trained model to:', backup_dir)

## 5. Zip and Download Model to Local Machine
Use this if you also want a direct file download from Colab (in addition to Drive backup).

In [ ]:
import os
from google.colab import files

zip_name = 't5_address_model.zip'
assert os.path.isdir('models/t5_address'), 'models/t5_address not found'

!zip -r {zip_name} models/t5_address
files.download(zip_name)